In [5]:
import pandas as pd 
from langchain_community.llms import Ollama
from langchain_classic.agents import AgentExecutor, create_structured_chat_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.tools import tool

In [6]:
#ici il nous faut une fonction qui transforme notre excel en data frame pandas
@tool
def lire_planning_excel(chemin_fichier: str) -> pd.DataFrame :
    """ Utile pour lire le contenu du fichier Excel """
    return pd.read_excel(chemin_fichier)

In [7]:
# Configuration d'Ollama et de l'Agent IA

llm = Ollama(model="llama3", temperature=0)

# Liste des outils mis à disposition de l'agent
tools = [lire_planning_excel]

# Création du prompt pour guider l'agent
prompt_path = "prompt_syst.txt"

with open(prompt_path, 'r') as file:
    system_prompt = file.read()


#on fait un joli prompt avec notre prompt de base auquel on ajoute de la mémoire pour qu'il garde l'historique de la conversation, inupt est notre question (humaine) et le blocnotes c'est là où l'agent écrit ses pensées
prompt = ChatPromptTemplate.from_messages([("system", system_prompt), MessagesPlaceholder(variable_name="chat_history", optional=True),("human", "{input}\n\n{agent_scratchpad}"),])

#on construit l'agent en lui donnant tout
agent = create_structured_chat_agent(llm, tools, prompt)

# on l"exécute ! 
agent_executor = AgentExecutor(
    agent=agent, 
    tools=tools, 
    verbose=True,  # pour voir le raisonnement
    handle_parsing_errors=True
)

/var/folders/z4/mxgvktxs5hjd5n8chzs4rn2r0000gn/T/ipykernel_7917/1047776095.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3", temperature=0)


ValueError: Prompt missing required variables: {'tools', 'tool_names'}